# Link-history features for the operation-time rungs

The static notebooks (01-09) predict a packet's path loss from what is known before deployment: geometry, frequency and the five sensors.
This notebook builds the inputs for the operation-time question: what will the next packet's path loss be, given the packets the gateway
has already received from that link? Every feature of a packet uses only earlier packets of the same link; the packet's own RSSI and SNR
are never used.

* Anchor `a`: the mean path loss of the link over the previous hour (fallbacks: the previous 60 packets, then the previous 20).
* Target `y = PL - a`: the deviation of the packet from its anchor. Models fitted on `y` carry no link identity unless a rung adds it.
* History features are expressed relative to the anchor (`r_*`), so a model trained on five links transfers to the sixth.
* Rungs H1-H7 add information in the order a network operator obtains it; notebooks 15-18 fit every model family on them.

Output: `Data_Files/history_train.csv`, `Data_Files/history_test.csv` (same rows and folds as `train.csv` / `test.csv`) and `Data_Files/history_rungs.json`.

In [1]:
import json
import numpy as np
import pandas as pd
from history_features import build, RUNGS, H7


In [2]:
# The chronological split and the expanding-window folds of notebook 00 (fold -1 = first training block, 0-4 = validation blocks, 9 = hold-out)
train = pd.read_csv("Data_Files/train.csv")
train["fold"] = np.load("Data_Files/train_folds.npy")
test = pd.read_csv("Data_Files/test.csv")
test["fold"] = 9
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["time"], utc=True, format="mixed")
d = d.drop(columns="time")
print(f"{len(train):,} training rows, {len(test):,} hold-out rows, {d.device_id.nunique()} links")


2,128,219 training rows, 532,055 hold-out rows, 6 links


In [3]:
# Causal history features and the anchor (history_features.py)
d = build(d)
print("features built")


features built


In [4]:
# Rungs: what the predictor knows, in the order a network obtains it (history_features.py)
rungs = RUNGS
with open("Data_Files/history_rungs.json", "w") as f:
    json.dump(rungs, f, indent=1)

usable = d["y"].notna() & d[H7].notna().all(axis=1)
print(pd.DataFrame({"inputs": {k: len(v) for k, v in rungs.items()}}))
print(f"rows with a complete H7 input and an anchor: {usable.mean() * 100:.2f} % (the rest are the first packets of each link and gaps)")
print("share of missing values per feature (%):")
print((d[H7].isna().mean() * 100).round(2).to_string())


                              inputs
H1 link history                   16
H2 + same-channel history         20
H3 + clock                        24
H4 + SF                           25
H5 + SNR of earlier packets       27
H6 + environmental sensors        33
H7 + link level and geometry      37
rows with a complete H7 input and an anchor: 99.94 % (the rest are the first packets of each link and gaps)
share of missing values per feature (%):


r_lag1         0.00
r_lag2         0.00
r_lag3         0.00
r_m5           0.00
r_m20          0.00
r_m6h          0.00
r_m24h         0.00
r_m7D          0.00
r_max60        0.00
r_min60        0.00
r_q95          0.05
s20            0.00
s1h            0.02
s24h           0.00
gap            0.00
n1h            0.01
r_c1           0.00
r_mc5          0.00
cgap           0.00
frequency      0.00
hsin           0.00
hcos           0.00
dow            0.00
work           0.00
SF             0.00
snr1           0.00
snr20          0.00
co2            0.00
humidity       0.00
pm25           0.00
pressure       0.00
temperature    0.00
co2ex          0.05
a              0.00
distance       0.00
c_walls        0.00
w_walls        0.00


In [5]:
# Save, same rows and folds as train.csv / test.csv
keep = ["device_id", "t", "fold", "SF", "snr", "PL", "distance", "frequency", "c_walls", "w_walls", "co2", "humidity", "pm25", "pressure", "temperature",
        "a", "y", "devmean", "work"] + [c for c in H7 if c not in ("SF", "distance", "frequency", "c_walls", "w_walls", "co2", "humidity", "pm25", "pressure", "temperature", "a", "work")]
d[d["fold"] != 9][keep].to_csv("Data_Files/history_train.csv", index=False, float_format="%.6g")
d[d["fold"] == 9][keep].to_csv("Data_Files/history_test.csv", index=False, float_format="%.6g")
print(f"saved {len(keep)} columns: Data_Files/history_train.csv ({(d['fold'] != 9).sum():,} rows), Data_Files/history_test.csv ({(d['fold'] == 9).sum():,} rows)")


saved 44 columns: Data_Files/history_train.csv (2,128,219 rows), Data_Files/history_test.csv (532,055 rows)
